# Hydro reservoir inflow from ERA5 climate reanalysis (PyPSA-Eur approach)

Replaces ENTSO-E's self-reported reservoir filling-rate as the source of natural inflow
for `Reservoir`/`PumpOpen` hydro storage. ENTSO-E's filling-rate series is missing for
11 of 25 modeled countries in 2024 (`BE, CZ, DK, DE, HU, IE, LU, NL, PL, SK, GB`),
including Germany's 772 GWh of pumped storage. Climate reanalysis data has no such gap —
every country gets a physically-grounded inflow series regardless of what its TSO chose
to report.

Method: ERA5 surface runoff, routed through river catchments (HydroBASINS) to each hydro
plant via `atlite`'s `cutout.hydro()`, calibrated to real annual Eurostat hydro generation
per country. Same idea as PyPSA-Eur's `scripts/build_hydro_profile.py` — see
https://github.com/PyPSA/pypsa-eur/blob/master/scripts/build_hydro_profile.py — adapted
here to use plant data from `parse_hydro_JRC_powerplants.ipynb` instead of PyPSA-Eur's own
plant list, and Eurostat instead of EIA for calibration (both already used elsewhere in
this repo).

**RunOfRiver is out of scope here** — it already has its own generation-based input
elsewhere in the pipeline and doesn't have a "storage inflow" in the same sense (no
reservoir to fill). This notebook only replaces the `Reservoir`/`PumpOpen` natural-inflow
calculation.

## Prerequisites — NOT satisfied in this environment as of writing; this notebook has
## been authored against atlite's and PyPSA-Eur's actual documented APIs but has **not**
## been executed. Before running it:

1. `pip install atlite geopandas` (neither is installed in the `base` conda env used by
   the rest of this pipeline as of 12.08.26).
2. Register at https://cds.climate.copernicus.eu and create `~/.cdsapirc` with your API
   credentials — follow CDS's *current* setup instructions rather than any snippet baked
   into this notebook, since the API changed in the 2024 CDS migration and an outdated
   snippet will silently fail. No `.cdsapirc` exists on this machine as of writing.
3. Download a HydroBASINS level covering Europe (PyPSA-Eur uses level 6) from
   https://www.hydrosheds.org/products/hydrobasins — registration/manual download in some
   regions — and save it under `../source_data/hydrobasins/`.
4. Budget time and disk: a full-year, hourly ERA5 cutout over this bounding box is a
   multi-GB download and the first `cutout.prepare()` call can take hours depending on
   the CDS queue.
5. Run `parse_hydro_JRC_powerplants.ipynb` first — this notebook reads its
   `hydro_plants_JRC_hpdb.csv` output.

Added by Claude (data lineage / hydro-gap fix session), 12.08.26 — written but unexecuted,
see the plan at the time (`golden-twirling-marshmallow.md`) for the verification steps to
run once the above is in place.

In [ ]:
import atlite
import pandas as pd
import numpy as np
import geopandas as gpd
import os

In [ ]:
baseyear = 2024

In [ ]:
dir_source = "../source_data/"
dir_out = "../parsed_data/"
cutout_dir = dir_source + "atlite_cutouts/"
os.makedirs(cutout_dir, exist_ok=True)

fn_plants = dir_out + "hydro_plants_JRC_hpdb.csv"          # from parse_hydro_JRC_powerplants.ipynb
fn_hydrobasins = dir_source + "hydrobasins/hybas_eu_lev06_v1c.shp"  # adjust to whichever level/region you download
fn_gen_eurostat = dir_out + "generation_yearly_eurostat.csv"
fn_additional = "../additional_data.xlsx"

fn_out = dir_out + f"hydro_natural_inflow_ERA5_atlite_hourly_{baseyear}.csv"

In [ ]:
df_countries = pd.read_excel(fn_additional, sheet_name="Countries_EU", index_col="Country")
countries = list(df_countries.index)

# gap countries confirmed this session (2024 reservoir_level_2024_weekly_entsoe_TP.csv
# has <10 non-null weekly observations) -- informational only, this notebook computes
# inflow for ALL 25 countries; Create_gdx_EU28_2024.ipynb decides which ones to actually use
known_gap_countries = ['BE', 'CZ', 'DK', 'DE', 'HU', 'IE', 'LU', 'NL', 'PL', 'SK', 'GB']

## Plant list and cutout bounding box

Compute the ERA5 cutout's bounding box from the plants' own lat/lon extremes (+ margin)
rather than hardcoding one, so upstream catchments just outside a country's borders
(e.g. Alpine basins feeding plants in multiple countries) are still covered.

In [ ]:
df_plants = pd.read_csv(fn_plants)
df_plants = df_plants.dropna(subset=["lat", "lon"]).set_index("id")
print("plants with valid coordinates:", len(df_plants))

margin = 2.0  # degrees
x_min, x_max = df_plants.lon.min() - margin, df_plants.lon.max() + margin
y_min, y_max = df_plants.lat.min() - margin, df_plants.lat.max() + margin
print(f"cutout bounding box: x=({x_min:.1f}, {x_max:.1f}) y=({y_min:.1f}, {y_max:.1f})")

# run-of-river plants are excluded here -- see notebook header. HDAM (dam/reservoir) and
# HPHS (pumped-storage, both open- and closed-loop -- the open/closed split happens
# downstream in Create_gdx_EU28_2024.ipynb using parse_hydro_JRC_powerplants.ipynb's
# capacity-weighted shares, same as it already does for the ENTSO-E-derived inflow)
# both receive natural inflow and are kept.
plants_hydro = df_plants[df_plants.type.isin(["HDAM", "HPHS"])].copy()
print("HDAM+HPHS plants (storage-inflow-relevant):", len(plants_hydro))
print(plants_hydro.groupby("country").size().sort_values(ascending=False).head(10))

## ERA5 cutout

`cutout.prepare()` is the step that actually hits the CDS API and downloads ERA5 data --
this is the multi-GB, multi-hour step referenced in the prerequisites above.

In [ ]:
cutout = atlite.Cutout(
    path=cutout_dir + f"europe-{baseyear}-era5.nc",
    module="era5",
    x=slice(x_min, x_max),
    y=slice(y_min, y_max),
    time=str(baseyear),
)
cutout.prepare(features=["runoff"])

## HydroBASINS catchment routing

In [ ]:
hydrobasins = gpd.read_file(fn_hydrobasins)
print("hydrobasins loaded:", len(hydrobasins), "basins")

In [ ]:
# atlite.convert.hydro(cutout, plants, hydrobasins, flowspeed=1, weight_with_height=False, ...)
# exposed as cutout.hydro(...); plants must have lon/lat columns (present in plants_hydro).
# flowspeed=1 m/s is atlite's own default (average water travel time from basin to plant).
inflow_da = cutout.hydro(plants_hydro, hydrobasins, flowspeed=1, show_progress=True)
# inflow_da: xarray DataArray, dims (plant, time), units MW-equivalent
df_inflow_plant = inflow_da.to_dataframe(name="MW").reset_index()
df_inflow_plant = df_inflow_plant.merge(
    plants_hydro[["country", "type"]], left_on="plant", right_index=True
)
df_inflow_plant.head()

In [ ]:
df_inflow_country = (
    df_inflow_plant.groupby(["country", "time"])["MW"]
    .sum()
    .reset_index()
    .rename(columns={"time": "date"})
)
df_inflow_country.head()

## Calibrate to Eurostat annual hydro generation

Same idea as PyPSA-Eur's `normalize_using_yearly=eia_stats`, done manually here since
we're working at country-aggregated level rather than through atlite's own
`cutout.runoff(normalize_using_yearly=...)` path.

**Assumption, stated explicitly:** this uses Eurostat's `Hydro` tech total (reservoir +
run-of-river generation) as a proxy for natural inflow over a full year -- it excludes
Eurostat's separate `Pump` category, since pumped-storage discharge draws on previously
*pumped* water, not fresh natural inflow, and including it would overstate the
calibration target. This still assumes turbine losses/curtailment/spillage roughly wash
out over a full year, the same assumption PyPSA-Eur's own calibration makes.

In [ ]:
df_gen = pd.read_csv(fn_gen_eurostat)
df_gen = df_gen[(df_gen.tech == "Hydro") & (df_gen.year == baseyear)].copy()
country_remap = {"EL": "GR", "UK": "GB"}  # Eurostat uses EL/UK; this pipeline uses GR/GB
df_gen["country"] = df_gen["country"].replace(country_remap)
annual_obs = df_gen.set_index("country")["MWh"]

annual_sim = df_inflow_country.groupby("country")["MW"].sum()  # hourly steps -> sum of MW = MWh

scale = (annual_obs / annual_sim).reindex(df_inflow_country.country.unique())
print("calibration scale factors:")
print(scale.sort_values())

extreme = scale[(scale.isna()) | (scale <= 0) | (scale > 10) | (scale < 0.1)]
print("\nextreme/undefined scale factors -- inspect before trusting:")
print(extreme)

In [ ]:
df_inflow_country["natural_inflow"] = df_inflow_country["MW"] * df_inflow_country["country"].map(scale)
df_inflow_country = df_inflow_country.dropna(subset=["natural_inflow"])

## Fallback if HydroBASINS acquisition stalls (documented, not run by default)

PyPSA-Eur's actual `build_hydro_profile.py` normally uses the simpler country-polygon
method, not per-plant catchment routing:

```python
# needs a country-shapes source (e.g. Natural Earth / GADM) instead of HydroBASINS +
# a per-plant list -- much easier to obtain, at the cost of catchment-level precision
cutout.runoff(
    shapes=country_shapes,
    smooth=True,
    lower_threshold_quantile=True,
    normalize_using_yearly=annual_obs,
)
```

Worth trying first if HydroBASINS turns out to be the harder acquisition step -- it skips
the plant list and catchment routing entirely and calibrates in the same call.

## Output

Columns (`date, country, natural_inflow`) deliberately match the existing
`df_reservoir_inflow_entsoe_hourly` in `Create_gdx_EU28_2024.ipynb`, so this is a
structural drop-in for the countries missing ENTSO-E data -- see the wiring change
there (`USE_ERA5_INFLOW` toggle).

In [ ]:
df_out = df_inflow_country[["date", "country", "natural_inflow"]].copy()
df_out.to_csv(fn_out, index=False, encoding="utf-8")
print(f"wrote {fn_out}")
df_out.head()

## Completeness diagnostic (run this after the above -- the smoke test referenced in the plan)

In [ ]:
print(f"{'country':8s} {'plants':>7s} {'sim_inflow_MWh':>16s} {'has_gen_target':>15s} {'hours_covered':>14s} {'gap_country':>12s}")
for c in countries:
    n_plants = (plants_hydro.country == c).sum()
    sim = annual_sim.get(c, 0.0)
    has_target = c in annual_obs.index
    hours = (df_out.country == c).sum()
    print(f"{c:8s} {n_plants:7d} {sim:16,.0f} {str(has_target):>15s} {hours:14d} {str(c in known_gap_countries):>12s}")